# Parking Space YOLO OBB Pipeline - Colab

Colab-ready pipeline for rotated parking-bay detection. It ignores XML boxes for OBB training and uses the RD New GeoJSON polygons as ground truth.

## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')


## 1. Install Dependencies

`ultralytics` is for YOLO OBB. `rasterio` reads GeoTIFF geotransforms. `shapely` handles polygon intersection and dedupe.

In [ ]:
!pip install -q ultralytics rasterio shapely pillow


## 2. Paths

Upload the project folder to `MyDrive/ParkingSpaceDetection`. OBB is only configured for round 2. Keep the filenames matched: `000000000041.tif`, `000000000041.png`, `000000000041.txt`.

In [ ]:
from pathlib import Path
import sys

BASE = Path('/content/gdrive/MyDrive/ParkingSpaceDetection')
ROUND = 'round2'
ROUND_DIR = BASE / 'data' / ROUND
OBJECT_DETECTION_MODELS = BASE / 'Object_Detection_Models'
WORKFLOW_SCRIPTS = OBJECT_DETECTION_MODELS / 'PythonOBB' / 'parking_obb_workflow'
RESULTS_DIR = BASE / 'results'

# OBB uses round 2 only. There is no round 1 OBB setup.
TIF_DIR = ROUND_DIR / 'images' / 'train_val'
PNG_DIR = ROUND_DIR / 'images' / 'train_val_png'
GEOJSON = BASE / 'Combine_Geojsons' / 'combined_annotations.geojson'
OBB_LABEL_DIR = ROUND_DIR / 'labels' / 'train_val_obb_txt'
DATASET_DIR = ROUND_DIR / 'obb_dataset'

RUNS_DIR = BASE / 'yolo11_obb_output'
TRAIN_PROJECT = RUNS_DIR
TRAIN_NAME = 'train'
TRAIN_DIR = TRAIN_PROJECT / TRAIN_NAME
BEST_MODEL = TRAIN_DIR / 'weights/best.pt'
LAST_MODEL = TRAIN_DIR / 'weights/last.pt'

# Directories test split.
TEST_DIR = BASE / 'data' / 'roundTest'
TEST_TIF_DIR = TEST_DIR / 'test_tiles'
TEST_PNG_DIR = TEST_DIR / 'images' / 'test'
TEST_GEOJSON = TEST_DIR / 'test_labels' / 'combined_annotations.geojson'

# Directories prediction split.
PREDICT_TIF = BASE / 'data' / 'predict'
PREDICT_PNG = BASE / 'data' / 'images' / 'predict'
PREDICT_SOURCE = PREDICT_PNG
PREDICT_PROJECT = RUNS_DIR
PREDICT_DEVICE = 0
PRED_GEOJSON = RESULTS_DIR / 'predictionsYoloV11_OBB.geojson'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('BASE:', BASE)
print('TIF_DIR:', TIF_DIR, TIF_DIR.exists())
print('GEOJSON:', GEOJSON, GEOJSON.exists())
print('WORKFLOW_SCRIPTS:', WORKFLOW_SCRIPTS, WORKFLOW_SCRIPTS.exists())
print('PREDICT_SOURCE:', PREDICT_SOURCE, PREDICT_SOURCE.exists())
print('TRAIN_DIR:', TRAIN_DIR)
print('BEST_MODEL:', BEST_MODEL)

## 3. Check Expected Files

In [ ]:
print('GeoTIFFs:', len(list(TIF_DIR.glob('*.tif'))))
print('Scripts:', sorted(p.name for p in WORKFLOW_SCRIPTS.glob('*.py')))


## 4. TIFF -> PNG

YOLO trains on PNG images. The GeoTIFFs stay in `images/` for georeferencing.

In [ ]:
!python {WORKFLOW_SCRIPTS / '01_tif_to_png.py'} \
  --input {TIF_DIR} \
  --output {PNG_DIR}


## 5. GeoJSON -> YOLO OBB Labels

This creates rotated labels from the GeoJSON polygons. XML is not used because XML only stores axis-aligned boxes.

In [ ]:
!python {WORKFLOW_SCRIPTS / '02_geojson_to_yolo_obb_no_dedup.py'} \
  --geojson {GEOJSON} \
  --tiles {TIF_DIR} \
  --output {OBB_LABEL_DIR} \
  --min-visible 0.90


## 6. Build Train/Val Dataset

In [ ]:
!python {WORKFLOW_SCRIPTS / '03_train_val_data_split.py'} \
  --images {PNG_DIR} \
  --labels {OBB_LABEL_DIR} \
  --dataset {DATASET_DIR} \
  --train-ratio 0.80 \
  --seed 42 \
  --min-labels-per-tile 1 \
  --overwrite


## 7. Inspect Dataset

In [ ]:
# Self-contained dataset check/build cell.
# If dataset/data.yaml does not exist yet, this cell runs the split step first.

print('PNG_DIR:', PNG_DIR)
print('OBB_LABEL_DIR:', OBB_LABEL_DIR)
print('DATASET_DIR:', DATASET_DIR)

pngs = sorted(PNG_DIR.glob('*.png'))
labels = sorted(OBB_LABEL_DIR.glob('*.txt'))
print('source pngs:', len(pngs))
print('source obb labels:', len(labels))

if not pngs:
    raise RuntimeError('No PNGs found. Run the TIFF -> PNG cell first.')
if not labels:
    raise RuntimeError('No OBB labels found. Run the GeoJSON -> YOLO OBB Labels cell first.')

if not (DATASET_DIR / 'data.yaml').exists():
    print('data.yaml missing, building train/val dataset now...')
    !python {WORKFLOW_SCRIPTS / '03_train_val_data_split.py'} \
      --images {PNG_DIR} \
      --labels {OBB_LABEL_DIR} \
      --dataset {DATASET_DIR} \
      --train-ratio 0.80 \
      --seed 42 \
      --min-labels-per-tile 1 \
      --overwrite

for folder in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    folder_path = DATASET_DIR / folder
    print(folder, len(list(folder_path.glob('*'))))

print('\ndata.yaml:')
print((DATASET_DIR / 'data.yaml').read_text())


## 8. Train YOLO OBB

Testing here is a fast CPU sanity run: `yolo11n-obb.pt`, 5 epochs, image size 640, batch 2. This is for testing that the pipeline works. For real training, switch to GPU and use `yolo11s-obb.pt`, more epochs, and `imgsz=1024`.


In [ ]:
# Fast sanity run. This should work on CPU, but it is only for testing the pipeline.
#TRAIN_MODEL = 'yolo11n-obb.pt'
#TRAIN_EPOCHS = 5
#TRAIN_IMGSZ = 640
#TRAIN_BATCH = 2
#TRAIN_DEVICE = 'cpu'

# For real training later, use something like:
TRAIN_MODEL = 'yolo11s-obb.pt'
TRAIN_EPOCHS = 200
TRAIN_IMGSZ = 1024
TRAIN_BATCH = 16
TRAIN_DEVICE = 0  # requires Colab GPU

print('training model:', TRAIN_MODEL)
print('training device:', TRAIN_DEVICE)
print('training output:', TRAIN_DIR)

!python {WORKFLOW_SCRIPTS / '04_train_obb.py'} \
  --data {DATASET_DIR / 'data.yaml'} \
  --model {TRAIN_MODEL} \
  --epochs {TRAIN_EPOCHS} \
  --imgsz {TRAIN_IMGSZ} \
  --batch {TRAIN_BATCH} \
  --device {TRAIN_DEVICE} \
  --project {TRAIN_PROJECT} \
  --name {TRAIN_NAME} \
  --exist-ok

## Check the details of settings of training/validation to report

In [ ]:
def count_label_rows(label_dir):
    total = 0
    for txt in label_dir.glob('*.txt'):
        total += len([line for line in txt.read_text().splitlines() if line.strip()])
    return total

train_instances = count_label_rows(DATASET_DIR / 'labels/train')
val_instances = count_label_rows(DATASET_DIR / 'labels/val')

print('train images:', len(list((DATASET_DIR / 'images/train').glob('*.png'))))
print('val images:', len(list((DATASET_DIR / 'images/val').glob('*.png'))))
print('train label files:', len(list((DATASET_DIR / 'labels/train').glob('*.txt'))))
print('val label files:', len(list((DATASET_DIR / 'labels/val').glob('*.txt'))))
print('train instances:', train_instances)
print('val instances:', val_instances)
print('total instances:', train_instances + val_instances)

## 9. Predict

Uses the trained `best.pt`. For the CPU sanity run, prediction also runs on CPU.


In [ ]:
!python {WORKFLOW_SCRIPTS / '05_predict_obb.py'} \
  --model {BEST_MODEL} \
  --source {PREDICT_SOURCE} \
  --project {PREDICT_PROJECT} \
  --name predict \
  --imgsz 1024 \
  --conf 0.3 \
  --device {PREDICT_DEVICE}

## 9b. Show Prediction Images

Ultralytics saves rendered prediction images in the predict folder. This cell displays them in Colab.


In [ ]:
# in order to get the font (installation)
!apt-get update -qq
!apt-get install -y fonts-dejavu-core

In [ ]:
from IPython.display import Image, display

predict_dirs = sorted(PREDICT_PROJECT.glob('predict*'), key=lambda p: p.stat().st_mtime)
PREDICT_DIR = predict_dirs[-1]

print('using:', PREDICT_DIR)

image_files = sorted(
    list(PREDICT_DIR.glob('*.jpg')) +
    list(PREDICT_DIR.glob('*.jpeg')) +
    list(PREDICT_DIR.glob('*.png'))
)

print('prediction images:', len(image_files))
for image_path in image_files[:10]:
    print(image_path.name)
    display(Image(filename=str(image_path)))

In [ ]:
from PIL import Image as PILImage, ImageDraw, ImageFont, ImageOps
from IPython.display import display

COMPARE_DIR = RESULTS_DIR / 'gt_vs_pred_preview_obb'
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

predict_dirs = sorted(PREDICT_PROJECT.glob('predict*'), key=lambda p: p.stat().st_mtime)
if not predict_dirs:
    raise RuntimeError(f'No predict folders found under {PREDICT_PROJECT}')

PREDICT_DIR = predict_dirs[-1]
PRED_TXT_DIR = PREDICT_DIR / 'labels'

print('using prediction folder:', PREDICT_DIR)
print('prediction labels dir:', PRED_TXT_DIR, PRED_TXT_DIR.exists())

try:
    title_font = ImageFont.truetype('DejaVuSans.ttf', 26)
except Exception:
    title_font = None

def draw_obb_file(draw, label_path, image_size, color, width=4):
    if not label_path.exists():
        return 0

    img_w, img_h = image_size
    count = 0

    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue

        parts = line.split()
        if len(parts) < 9:
            continue

        coords = [float(v) for v in parts[1:9]]
        points = [
            (coords[i] * img_w, coords[i + 1] * img_h)
            for i in range(0, 8, 2)
        ]

        # Draw on the original image first, so boxes are clipped at image edges.
        draw.line(points + [points[0]], fill=color, width=width)
        count += 1

    return count

def get_text_width(draw, text, font):
    if hasattr(draw, 'textbbox'):
        box = draw.textbbox((0, 0), text, font=font)
        return box[2] - box[0]
    return draw.textlength(text, font=font)

val_images = sorted((DATASET_DIR / 'images/val').glob('*.png'))
print('val images:', len(val_images))

TITLE_MARGIN = 30

for image_path in val_images:
    base_img = PILImage.open(image_path).convert('RGB')
    base_draw = ImageDraw.Draw(base_img)

    gt_path = DATASET_DIR / 'labels/val' / f'{image_path.stem}.txt'
    pred_path = PRED_TXT_DIR / f'{image_path.stem}.txt'

    # Style: GT red, prediction blue.
    gt_count = draw_obb_file(base_draw, gt_path, base_img.size, color='red', width=4)
    pred_count = draw_obb_file(base_draw, pred_path, base_img.size, color='blue', width=4)

    # Add title margin after drawing boxes, so boxes cannot enter the title area.
    img = ImageOps.expand(base_img, border=(0, TITLE_MARGIN, 0, 0), fill='white')
    draw = ImageDraw.Draw(img)

    title = f'Image: {image_path.name} (GT: Red, Pred: Blue)'
    title_x = int((img.size[0] - get_text_width(draw, title, title_font)) / 2)
    title_y = 4

    draw.text((title_x, title_y), title, fill='black', font=title_font)

    out_path = COMPARE_DIR / image_path.name
    img.save(out_path)

    print(out_path.name, 'GT:', gt_count, 'pred:', pred_count)
    display(img)

## 10. Predictions -> RD New GeoJSON

Prediction `.txt` files are converted back to RD New polygons using the same-name GeoTIFF.

In [ ]:
predict_dirs = sorted(PREDICT_PROJECT.glob('predict*'), key=lambda p: p.stat().st_mtime)
if not predict_dirs:
    raise RuntimeError(f'No predict folders found under {PREDICT_PROJECT}')

PREDICT_DIR = predict_dirs[-1]
PRED_TXT_DIR = PREDICT_DIR / 'labels'
print('using prediction labels:', PRED_TXT_DIR)

!python {WORKFLOW_SCRIPTS / '06_predictions_to_geojson.py'} \
  {PRED_TXT_DIR} \
  --tiles {TIF_DIR} \
  -o {PRED_GEOJSON}

# 11. Test

In [ ]:
# Test paths are defined in the path cell.
print('TEST_TIF_DIR:', TEST_TIF_DIR)
print('TEST_PNG_DIR:', TEST_PNG_DIR)


In [ ]:
TEST_PNG_DIR.mkdir(parents=True, exist_ok=True)

!python "{WORKFLOW_SCRIPTS / '01_tif_to_png.py'}" \
  --input "{TEST_TIF_DIR}" \
  --output "{TEST_PNG_DIR}"

print('test tif count:', len(list(TEST_TIF_DIR.glob('*.tif'))))
print('test png count:', len(list(TEST_PNG_DIR.glob('*.png'))))
print(sorted([p.name for p in TEST_PNG_DIR.glob('*.png')])[:10])

In [ ]:
from pathlib import Path
import shutil
from ultralytics import YOLO
import torch

TEST_EVAL_DIR = TEST_DIR / 'dataset_eval'
TEST_EVAL_IMG_DIR = TEST_EVAL_DIR / 'images/val'
TEST_EVAL_LABEL_DIR = TEST_EVAL_DIR / 'labels/val'
TRAIN_DEVICE = 0 if torch.cuda.is_available() else 'cpu'

print('TEST_GEOJSON:', TEST_GEOJSON)
print('TEST_EVAL_DIR:', TEST_EVAL_DIR)


In [ ]:
# Rebuild clean YOLO evaluation folder
if TEST_EVAL_DIR.exists():
    shutil.rmtree(TEST_EVAL_DIR)

TEST_EVAL_IMG_DIR.mkdir(parents=True, exist_ok=True)
TEST_EVAL_LABEL_DIR.mkdir(parents=True, exist_ok=True)

# Copy test PNGs into YOLO eval image folder
for img in sorted(TEST_PNG_DIR.glob('*.png')):
    shutil.copy2(img, TEST_EVAL_IMG_DIR / img.name)

# Generate YOLO OBB labels from test GeoJSON + test GeoTIFFs
!python "{WORKFLOW_SCRIPTS / '02_geojson_to_yolo_obb_no_dedup.py'}" \
  --geojson "{TEST_GEOJSON}" \
  --tiles "{TEST_TIF_DIR}" \
  --output "{TEST_EVAL_LABEL_DIR}" \
  --min-visible 0.90

# Write YOLO data.yaml for independent test evaluation
data_yaml = TEST_EVAL_DIR / 'data.yaml'
data_yaml.write_text(f"""path: {TEST_EVAL_DIR}
train: images/val
val: images/val

names:
  0: parking_space
""")

print('test images:', len(list(TEST_EVAL_IMG_DIR.glob('*.png'))))
print('test labels:', len(list(TEST_EVAL_LABEL_DIR.glob('*.txt'))))
print('data.yaml:', data_yaml)

# Validate trained best.pt on independent test set
model = YOLO(str(BEST_MODEL))

metrics = model.val(
    data=str(data_yaml),
    imgsz=1024,
    batch=16,
    device=TRAIN_DEVICE,
    single_cls=True,
    rect=True,
    split='val',
    project=str(TEST_DIR / 'runs/obb'),
    name='test_val',
    exist_ok=True,
)

print('\nTest metrics:')
print('Precision:', metrics.results_dict.get('metrics/precision(B)'))
print('Recall:', metrics.results_dict.get('metrics/recall(B)'))
print('mAP50:', metrics.results_dict.get('metrics/mAP50(B)'))
print('mAP50-95:', metrics.results_dict.get('metrics/mAP50-95(B)'))
print('results saved to:', TEST_DIR / 'runs/obb/test_val')

# 11b Preview of Predictions of Test areas

In [ ]:
from pathlib import Path
from PIL import Image as PILImage, ImageDraw, ImageFont, ImageOps
from IPython.display import display

# Correct folders for test preview
TEST_EVAL_DIR = TEST_DIR / 'dataset_eval'
TEST_COMPARE_DIR = TEST_DIR / 'gt_vs_pred_preview'
TEST_PRED_PROJECT = TEST_DIR / 'runs/obb'
TEST_PRED_NAME = 'predict_test'
TEST_PRED_LABEL_DIR = TEST_PRED_PROJECT / TEST_PRED_NAME / 'labels'

TEST_IMAGE_DIR = TEST_EVAL_DIR / 'images/val'
TEST_GT_LABEL_DIR = TEST_EVAL_DIR / 'labels/val'

TEST_COMPARE_DIR.mkdir(parents=True, exist_ok=True)

print('test images:', TEST_IMAGE_DIR, TEST_IMAGE_DIR.exists())
print('GT labels:', TEST_GT_LABEL_DIR, TEST_GT_LABEL_DIR.exists())
print('prediction labels:', TEST_PRED_LABEL_DIR, TEST_PRED_LABEL_DIR.exists())

if not TEST_IMAGE_DIR.exists():
    raise FileNotFoundError(TEST_IMAGE_DIR)
if not TEST_GT_LABEL_DIR.exists():
    raise FileNotFoundError(TEST_GT_LABEL_DIR)
if not TEST_PRED_LABEL_DIR.exists():
    raise FileNotFoundError(TEST_PRED_LABEL_DIR)

try:
    title_font = ImageFont.truetype('DejaVuSans', 26)
except Exception:
    title_font = None

def draw_obb_txt(draw, label_path, image_size, color, width=4):
    if not label_path.exists():
        return 0

    img_w, img_h = image_size
    count = 0

    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 9:
            continue

        coords = [float(v) for v in parts[1:9]]
        points = [
            (coords[i] * img_w, coords[i + 1] * img_h)
            for i in range(0, 8, 2)
        ]

        # Draw before adding title margin, so boxes clip at image edges.
        draw.line(points + [points[0]], fill=color, width=4)
        count += 1

    return count

def get_text_width(draw, text, font):
    if hasattr(draw, 'textbbox'):
        box = draw.textbbox((0, 0), text, font=font)
        return box[2] - box[0]
    return draw.textlength(text, font=font)

test_images = sorted(TEST_IMAGE_DIR.glob('*.png'))
print('test image count:', len(test_images))

TITLE_MARGIN = 30

for image_path in test_images[:12]:
    base_img = PILImage.open(image_path).convert('RGB')
    base_draw = ImageDraw.Draw(base_img)

    gt_path = TEST_GT_LABEL_DIR / f'{image_path.stem}.txt'
    pred_path = TEST_PRED_LABEL_DIR / f'{image_path.stem}.txt'

    gt_count = draw_obb_txt(base_draw, gt_path, base_img.size, color='red', width=4)
    pred_count = draw_obb_txt(base_draw, pred_path, base_img.size, color='blue', width=4)

    img = ImageOps.expand(base_img, border=(0, TITLE_MARGIN, 0, 0), fill='white')
    draw = ImageDraw.Draw(img)

    title = f'Image: {image_path.name} (GT: Red, Pred: Blue)'
    title_x = int((img.size[0] - get_text_width(draw, title, title_font)) / 2)
    title_y = 0
    draw.text((title_x, title_y), title, fill='black', font=title_font)

    out_path = TEST_COMPARE_DIR / image_path.name
    img.save(out_path)

    print(image_path.name, 'GT:', gt_count, 'pred:', pred_count)
    display(img)

In [ ]:
from pathlib import Path

TEST_PRED_DIR = TEST_PRED_PROJECT / TEST_PRED_NAME
TEST_PRED_TXT_DIR = TEST_PRED_DIR / 'labels'
TEST_PRED_GEOJSON = TEST_DIR / 'predictions_OBB.geojson'


print('TEST_PRED_DIR:', TEST_PRED_DIR)
print('TEST_PRED_DIR exists:', TEST_PRED_DIR.exists())

print('TEST_PRED_TXT_DIR:', TEST_PRED_TXT_DIR)
print('TEST_PRED_TXT_DIR exists:', TEST_PRED_TXT_DIR.exists())
print('prediction txt count:', len(list(TEST_PRED_TXT_DIR.glob('*.txt'))) if TEST_PRED_TXT_DIR.exists() else 'missing')

import subprocess

PRED_TO_GEOJSON_SCRIPT = WORKFLOW_SCRIPTS / '06_predictions_to_geojson.py'

cmd = [
    'python',
    str(PRED_TO_GEOJSON_SCRIPT),
    str(TEST_PRED_TXT_DIR),
    '--tiles', str(TEST_TIF_DIR),
    '-o', str(TEST_PRED_GEOJSON),
]

print('running:')
print(' '.join(cmd))

subprocess.run(cmd, check=True)

print('GeoJSON saved:', TEST_PRED_GEOJSON)
print('exists:', TEST_PRED_GEOJSON.exists())

# 12 Prediction on the neighbourhood, Nude in Wageningen

In [ ]:
# Define prediction output paths.
PREDICT_NAME = 'predict_nude'
PREDICT_DIR = PREDICT_PROJECT / PREDICT_NAME
PREDICT_TXT_DIR = PREDICT_DIR / 'labels'
PREDICT_GEOJSON = RESULTS_DIR / 'predictions_nude_OBB.geojson'


PREDICT_SCRIPT = WORKFLOW_SCRIPTS / '05_predict_obb.py'
PRED_TO_GEOJSON_SCRIPT = WORKFLOW_SCRIPTS / '06_predictions_to_geojson.py'

print('PREDICT_TIF:', PREDICT_TIF, PREDICT_TIF.exists())
print('PREDICT_PNG:', PREDICT_PNG)
print('BEST_MODEL:', BEST_MODEL, BEST_MODEL.exists())
print('PREDICT_SCRIPT:', PREDICT_SCRIPT, PREDICT_SCRIPT.exists())
print('PRED_TO_GEOJSON_SCRIPT:', PRED_TO_GEOJSON_SCRIPT, PRED_TO_GEOJSON_SCRIPT.exists())
print('prediction folder:', PREDICT_DIR)
print('output geojson:', PREDICT_GEOJSON)

In [ ]:
# Convert Nude TIFFs To PNGs
import subprocess

PREDICT_PNG.mkdir(parents=True, exist_ok=True)

cmd = [
    'python',
    str(WORKFLOW_SCRIPTS / '01_tif_to_png.py'),
    '--input', str(PREDICT_TIF),
    '--output', str(PREDICT_PNG),
]

print('running:')
print(' '.join(cmd))

subprocess.run(cmd, check=True)

print('prediction tif count:', len(list(PREDICT_TIF.glob('*.tif'))))
print('prediction png count:', len(list(PREDICT_PNG.glob('*.png'))))

In [ ]:
# Run Prediction On Nude
import shutil
import subprocess

if PREDICT_DIR.exists():
    shutil.rmtree(PREDICT_DIR)

cmd = [
    'python',
    str(PREDICT_SCRIPT),
    '--model', str(BEST_MODEL),
    '--source', str(PREDICT_PNG),
    '--project', str(PREDICT_PROJECT),
    '--name', PREDICT_NAME,
    '--imgsz', '1024',
    '--conf', '0.3',
    '--device', '0',
]

print('running:')
print(' '.join(cmd))

subprocess.run(cmd, check=True)

pred_images = (
    list(PREDICT_DIR.glob('*.png')) +
    list(PREDICT_DIR.glob('*.jpg')) +
    list(PREDICT_DIR.glob('*.jpeg'))
)

print('prediction images:', len(pred_images))
print('prediction labels:', len(list(PREDICT_TXT_DIR.glob('*.txt'))))
print('prediction folder:', PREDICT_DIR)

# 12b Preview of Prediction for Nude Neighbourhood

In [ ]:
# Show Prediction Preview Images
from IPython.display import Image, display

preview_images = sorted(
    list(PREDICT_DIR.glob('*.png')) +
    list(PREDICT_DIR.glob('*.jpg')) +
    list(PREDICT_DIR.glob('*.jpeg'))
)[:12]

print('showing:', len(preview_images))

for image_path in preview_images:
    print(image_path.name)
    display(Image(filename=str(image_path)))

In [ ]:
# Convert Nude Predictions To GeoJSON
import subprocess

cmd = [
    'python',
    str(PRED_TO_GEOJSON_SCRIPT),
    str(PREDICT_TXT_DIR),
    '--tiles', str(PREDICT_TIF),
    '-o', str(PREDICT_GEOJSON),
]

print('running:')
print(' '.join(cmd))

subprocess.run(cmd, check=True)

print('GeoJSON saved to:', PREDICT_GEOJSON)
print('exists:', PREDICT_GEOJSON.exists())

## Notes

- CRS is `EPSG:28992 - Amersfoort / RD New`.
- Upload required round 2 data to `MyDrive/ParkingSpaceDetection/data/round2/...`.
- OBB training uses only `data/round2`; there is no round 1 OBB workflow.
- Training outputs are saved in Google Drive at `yolo11_obb_output/train/`.
- The trained parking-space checkpoint is `yolo11_obb_output/train/weights/best.pt`.
- Prediction outputs are saved in Google Drive at `yolo11_obb_output/predict*/`.
- Validation outputs are saved in Google Drive at `yolo11_obb_output/val/`.
- The XML labels are optional for this notebook and are not used for OBB.
- The important link is the basename: `000000000041.png`, `000000000041.tif`, `000000000041.txt`.